In [1]:
"""
================================================================================
  LGG Brain MRI Segmentation — LEAKY EVALUATION VERSION (FIXED)
  Purpose: Reproduce data leakage methodology used in papers like
           LiteMRINet (PLOS ONE 2025) for direct scientific comparison.

  KEY DIFFERENCES FROM v3 (intentional):
    ✗  Random slice-level split (NO patient grouping)  ← LEAKY
    ✗  Single 3000/929 split, no cross-validation      ← LEAKY
    ✓  Same architecture as v3 (Attention Res-UNet 4.8M params)
    ✓  Same loss, optimizer, augmentation

  SCIENTIFIC PURPOSE:
    Compare inflated (leaky) vs honest (GroupKFold) metrics to quantify
    how much data leakage inflates published results.

  FIXES vs previous version:
    - MIXED_PRECISION = False  →  eliminates f16/f32 residual conflict
    - drop_remainder = True    →  eliminates partial batch shape error
    - Slice-by-slice inference →  no batch size issue in evaluation
================================================================================
"""

import os, gc, glob, math, random, warnings, logging, zipfile
import numpy as np
import pandas as pd
import cv2
from scipy.ndimage import label as scipy_label
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, Model, Input, backend as K
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, CSVLogger, LambdaCallback
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

# ─────────────────────────────────────────────────────────────────────────────
# 1.  CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
CFG = dict(
    DATA_ROOT        = "./kaggle_3m",
    ZIP_PATH         = "kaggle_3m.zip",
    OUTPUT_DIR       = "./outputs_leaky",

    IMG_SIZE         = 128,
    INPUT_CHANNELS   = 3,

    # ── LEAKY SPLIT ───────────────────────────────────────────────────────────
    TEST_SIZE        = 0.236,   # 929/3929 ≈ same ratio as LiteMRINet
    RANDOM_SEED      = 42,
    # NO patient grouping — intentionally leaky
    # ─────────────────────────────────────────────────────────────────────────

    BATCH_SIZE       = 16,
    EPOCHS           = 120,
    WARMUP_EPOCHS    = 8,
    LR_INITIAL       = 3e-4,
    LR_MIN           = 1e-6,
    WEIGHT_DECAY     = 1e-4,
    GRAD_CLIP        = 1.0,
    SEED             = 42,

    # ── CRITICAL FIX: mixed precision OFF ────────────────────────────────────
    # Mixed precision causes f16/f32 conflicts in residual Add layers with XLA.
    # v3 worked because GroupKFold naturally had larger balanced batches.
    # For this controlled experiment we use float32 throughout.
    MIXED_PRECISION  = False,
    # ─────────────────────────────────────────────────────────────────────────

    BASE_FILTERS     = 32,
    GROUP_NORM_G     = 8,
    DROPOUT          = 0.25,
    DS_WEIGHT        = 0.30,

    FT_ALPHA         = 0.80,
    FT_BETA          = 0.20,
    FT_GAMMA         = 0.60,
    BF_GAMMA         = 2.0,
    LOSS_FT_W        = 0.70,
    LOSS_BF_W        = 0.30,
    LABEL_SMOOTH     = 0.02,

    TUMOR_OVERSAMPLE = 2.0,
    EMPTY_KEEP_FRAC  = 0.40,

    AUG_PROB_TUMOR   = 0.90,
    AUG_PROB_EMPTY   = 0.20,
    ROT_RANGE        = 18,
    ZOOM_MIN         = 0.85,
    ZOOM_MAX         = 1.15,
    ELASTIC_ALPHA    = 32,
    ELASTIC_SIGMA    = 4,
    BRIGHTNESS_DELTA = 0.25,
    CONTRAST_RANGE   = (0.80, 1.20),
    NOISE_STD        = 0.03,

    MIN_COMPONENT_PX = 20,
    ES_PATIENCE      = 25,
)

os.makedirs(CFG["OUTPUT_DIR"], exist_ok=True)
SEED = CFG["SEED"]
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
IMG_SIZE = CFG["IMG_SIZE"]

# Float32 policy — no mixed precision for this script
tf.keras.mixed_precision.set_global_policy("float32")
print("Mixed precision: DISABLED (float32 throughout)")


# ─────────────────────────────────────────────────────────────────────────────
# 2.  DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
def build_dataframe(data_root, zip_path):
    if not os.path.exists(data_root):
        if os.path.exists(zip_path):
            print(f"Extracting {zip_path}...")
            with zipfile.ZipFile(zip_path, "r") as z:
                z.extractall(data_root)
        else:
            raise FileNotFoundError("No data found.")

    mask_files = sorted(
        glob.glob(os.path.join(data_root, "**", "*_mask.tif"), recursive=True))
    if not mask_files:
        raise ValueError("No *_mask.tif found.")

    records = []
    for msk_path in mask_files:
        img_path   = msk_path.replace("_mask", "")
        patient_id = os.path.basename(os.path.dirname(msk_path))
        if not os.path.exists(img_path):
            continue
        msk       = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)
        has_tumor = (msk is not None) and (int(msk.max()) > 0)
        tumor_frac = float(msk.astype(bool).sum()) / msk.size if has_tumor else 0.0
        records.append(dict(
            image_path=img_path, mask_path=msk_path,
            patient_id=patient_id, has_tumor=has_tumor,
            tumor_frac=tumor_frac))

    df = pd.DataFrame(records)
    print(f"Dataset: {len(df)} slices | "
          f"{df.has_tumor.sum()} tumor+ | "
          f"{df.patient_id.nunique()} patients")
    return df.reset_index(drop=True)


# ─────────────────────────────────────────────────────────────────────────────
# 3.  LEAKY SPLIT — random slice-level, no patient grouping
# ─────────────────────────────────────────────────────────────────────────────
def make_leaky_split(df):
    df_train, df_test = train_test_split(
        df, test_size=CFG["TEST_SIZE"],
        random_state=CFG["RANDOM_SEED"], shuffle=True)

    train_patients  = set(df_train["patient_id"])
    test_patients   = set(df_test["patient_id"])
    leaked_patients = train_patients & test_patients
    total_patients  = len(train_patients | test_patients)

    print(f"\n{'='*62}")
    print(f"  LEAKY SPLIT STATISTICS (replicating published papers)")
    print(f"{'='*62}")
    print(f"  Train slices : {len(df_train)}")
    print(f"  Test slices  : {len(df_test)}")
    print(f"  Total patients        : {total_patients}")
    print(f"  Patients train only   : {len(train_patients - test_patients)}")
    print(f"  Patients test only    : {len(test_patients - train_patients)}")
    print(f"  *** LEAKED patients   : {len(leaked_patients)} ***")
    print(f"  *** Leakage rate      : "
          f"{100*len(leaked_patients)/total_patients:.1f}% of all patients ***")
    print(f"{'='*62}\n")

    report_path = os.path.join(CFG["OUTPUT_DIR"], "leakage_report.txt")
    os.makedirs(CFG["OUTPUT_DIR"], exist_ok=True)
    with open(report_path, "w") as f:
        f.write("LEAKY SPLIT REPORT\n" + "="*50 + "\n")
        f.write(f"Train slices: {len(df_train)}\n")
        f.write(f"Test slices: {len(df_test)}\n")
        f.write(f"Total patients: {total_patients}\n")
        f.write(f"Leaked patients: {len(leaked_patients)}\n")
        f.write(f"Leakage rate: {100*len(leaked_patients)/total_patients:.1f}%\n\n")
        f.write("Leaked patient IDs:\n")
        for pid in sorted(leaked_patients):
            tr_n = (df_train["patient_id"] == pid).sum()
            te_n = (df_test["patient_id"] == pid).sum()
            f.write(f"  {pid}: {tr_n} train, {te_n} test slices\n")
    print(f"  Leakage report: {report_path}")
    return df_train.reset_index(drop=True), df_test.reset_index(drop=True)


def make_balanced_train_df(df):
    tumor_df = df[df.has_tumor].copy()
    empty_df = df[~df.has_tumor].copy()
    n_over   = int(len(tumor_df) * CFG["TUMOR_OVERSAMPLE"])
    tumor_up = tumor_df.sample(n=n_over, replace=True, random_state=SEED)
    n_empty  = int(len(empty_df) * CFG["EMPTY_KEEP_FRAC"])
    empty_dn = empty_df.sample(n=n_empty, replace=False, random_state=SEED)
    balanced = pd.concat([tumor_up, empty_dn]).sample(
        frac=1., random_state=SEED).reset_index(drop=True)
    print(f"  Balanced train: {len(balanced)} slices "
          f"(tumor {100*balanced.has_tumor.mean():.1f}%)")
    return balanced


# ─────────────────────────────────────────────────────────────────────────────
# 4.  AUGMENTATION
# ─────────────────────────────────────────────────────────────────────────────
def _rotate(img, msk):
    a = np.random.uniform(-CFG["ROT_RANGE"], CFG["ROT_RANGE"])
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), a, 1.0)
    img = cv2.warpAffine(img, M, (w,h),
                         flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
    msk = cv2.warpAffine(msk, M, (w,h),
                         flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_REFLECT_101)
    return img, msk

def _zoom(img, msk):
    s = np.random.uniform(CFG["ZOOM_MIN"], CFG["ZOOM_MAX"])
    h, w = img.shape[:2]
    nh, nw = int(h*s), int(w*s)
    ir = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    mr = cv2.resize(msk, (nw, nh), interpolation=cv2.INTER_NEAREST)
    if s > 1.0:
        y0, x0 = (nh-h)//2, (nw-w)//2
        ir, mr = ir[y0:y0+h, x0:x0+w], mr[y0:y0+h, x0:x0+w]
    else:
        ph, pw = h-nh, w-nw
        t, b = ph//2, ph-ph//2
        l, r = pw//2, pw-pw//2
        ir = cv2.copyMakeBorder(ir, t, b, l, r, cv2.BORDER_REFLECT_101)
        mr = cv2.copyMakeBorder(mr, t, b, l, r, cv2.BORDER_CONSTANT, value=0)
    return ir, mr

def _elastic(img, msk):
    a, s = CFG["ELASTIC_ALPHA"], CFG["ELASTIC_SIGMA"]
    h, w = img.shape[:2]
    dx = cv2.GaussianBlur(np.random.randn(h,w).astype(np.float32)*a, (0,0), s)
    dy = cv2.GaussianBlur(np.random.randn(h,w).astype(np.float32)*a, (0,0), s)
    x, y = np.meshgrid(np.arange(w), np.arange(h))
    img = cv2.remap(img, (x+dx).astype(np.float32), (y+dy).astype(np.float32),
                    cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
    msk = cv2.remap(msk, (x+dx).astype(np.float32), (y+dy).astype(np.float32),
                    cv2.INTER_NEAREST, borderMode=cv2.BORDER_REFLECT_101)
    return img, msk

def _photometric(img):
    img = np.clip(img + np.random.uniform(
        -CFG["BRIGHTNESS_DELTA"], CFG["BRIGHTNESS_DELTA"]), 0., 1.)
    c = np.random.uniform(*CFG["CONTRAST_RANGE"])
    img = np.clip((img - 0.5)*c + 0.5, 0., 1.)
    if np.random.rand() < 0.40:
        img = np.clip(
            img + np.random.randn(*img.shape).astype(np.float32)*CFG["NOISE_STD"],
            0., 1.)
    return img

def augment(img, msk, has_tumor):
    p = CFG["AUG_PROB_TUMOR"] if has_tumor else CFG["AUG_PROB_EMPTY"]
    if np.random.rand() > p:
        return img, msk
    if np.random.rand() < 0.75: img, msk = _rotate(img, msk)
    if np.random.rand() < 0.55: img, msk = _zoom(img, msk)
    if np.random.rand() < 0.40: img, msk = _elastic(img, msk)
    img = _photometric(img)
    return img, msk


# ─────────────────────────────────────────────────────────────────────────────
# 5.  tf.data PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
LABEL_SMOOTH = CFG["LABEL_SMOOTH"]

def _py_load(ip_t, mp_t, ht_t, training):
    ip = ip_t.numpy().decode(); mp = mp_t.numpy().decode()
    has_tumor = bool(ht_t.numpy())
    img = cv2.imread(ip, cv2.IMREAD_COLOR)
    if img is None:
        img = np.zeros((IMG_SIZE, IMG_SIZE, 3), np.float32)
    else:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), cv2.INTER_LINEAR)
        img = img.astype(np.float32) / 255.0
    msk = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
    if msk is None:
        msk = np.zeros((IMG_SIZE, IMG_SIZE), np.float32)
    else:
        msk = cv2.resize(msk, (IMG_SIZE, IMG_SIZE), cv2.INTER_NEAREST)
        msk = (msk > 127).astype(np.float32)
    if training:
        img, msk = augment(img, msk, has_tumor)
        msk = np.clip(msk*(1.-2*LABEL_SMOOTH)+LABEL_SMOOTH, 0., 1.)
    msk = msk[..., np.newaxis].astype(np.float32)
    return img.astype(np.float32), msk

def _tf_load(ip, mp, ht, training):
    img, msk = tf.py_function(
        func=lambda i,m,h: _py_load(i,m,h,training),
        inp=[ip, mp, ht], Tout=[tf.float32, tf.float32])
    img.set_shape([IMG_SIZE, IMG_SIZE, CFG["INPUT_CHANNELS"]])
    msk.set_shape([IMG_SIZE, IMG_SIZE, 1])
    return img, msk

def build_dataset(df, training, batch_size):
    ds = tf.data.Dataset.from_tensor_slices((
        df["image_path"].values,
        df["mask_path"].values,
        df["has_tumor"].values.astype(np.int32),
    ))
    if training:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda i,m,h: _tf_load(i,m,h,training),
                num_parallel_calls=tf.data.AUTOTUNE)
    # drop_remainder=True for BOTH splits → avoids partial-batch shape errors
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


# ─────────────────────────────────────────────────────────────────────────────
# 6.  ARCHITECTURE — Attention Res-UNet (identical to v3)
# ─────────────────────────────────────────────────────────────────────────────
class GroupNorm(layers.Layer):
    def __init__(self, groups=8, eps=1e-5, **kw):
        super().__init__(**kw)
        self.groups = groups
        self.eps    = eps

    def build(self, shape):
        C = int(shape[-1])
        self.gamma = self.add_weight(
            name="g", shape=(1,1,1,C), initializer="ones")
        self.beta  = self.add_weight(
            name="b", shape=(1,1,1,C), initializer="zeros")

    def call(self, x):
        B, H, W, C = tf.unstack(tf.shape(x))
        G  = self.groups
        xr = tf.reshape(x, [B, H, W, G, C//G])
        mu, va = tf.nn.moments(xr, axes=[1,2,4], keepdims=True)
        xr = (xr - mu) / tf.sqrt(va + self.eps)
        return tf.reshape(xr, [B, H, W, C]) * self.gamma + self.beta

    def get_config(self):
        c = super().get_config()
        c.update({"groups": self.groups, "eps": self.eps})
        return c


class StochasticDepth(layers.Layer):
    """DropPath — no mixed precision, uses input dtype directly."""
    def __init__(self, drop_prob=0.10, **kw):
        super().__init__(**kw)
        self.drop_prob = drop_prob

    def call(self, x, training=None):
        if (not training) or self.drop_prob == 0.:
            return x
        keep  = 1. - self.drop_prob
        # Build mask shape dynamically matching x rank → [B,1,1,...,1]
        ndim  = len(x.shape)
        shape = tf.concat([[tf.shape(x)[0]],
                           tf.ones([ndim - 1], dtype=tf.int32)], axis=0)
        # Cast keep to same dtype as x to avoid f16/f32 conflict
        keep_t = tf.cast(keep, x.dtype)
        mask   = tf.floor(keep_t + tf.cast(
            tf.random.uniform(shape), x.dtype))
        return x / keep_t * mask

    def get_config(self):
        c = super().get_config()
        c.update({"drop_prob": self.drop_prob})
        return c


def res_block(x, filters, groups=8, drop_path=0.):
    shortcut = x
    if x.shape[-1] != filters:
        shortcut = layers.Conv2D(
            filters, 1, padding="same", use_bias=False)(x)
        shortcut = GroupNorm(groups)(shortcut)
        
    # خط tf.cast حذف شد چون تایپ پیش‌فرض float32 است
    
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = GroupNorm(groups)(x)
    x = layers.Activation("gelu")(x)
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = GroupNorm(groups)(x)
    
    if drop_path > 0:
        x = StochasticDepth(drop_path)(x)
        
    # خط tf.cast دوم هم حذف شد
    
    x = layers.Add()([x, shortcut])
    x = layers.Activation("gelu")(x)
    return x


def attn_gate(g, s, f):
    g1  = layers.Conv2D(f, 1, padding="same")(g)
    s1  = layers.Conv2D(f, 1, padding="same")(s)
    psi = layers.Activation("relu")(layers.Add()([g1, s1]))
    psi = layers.Conv2D(1, 1, padding="same")(psi)
    psi = layers.Activation("sigmoid")(psi)
    return layers.Multiply()([s, psi])


def ds_head(x, target, name):
    x = layers.Conv2D(1, 1, padding="same")(x)
    x = layers.Resizing(target, target,
                        interpolation="bilinear", name=name)(x)
    return layers.Activation("sigmoid", dtype="float32",
                             name=name + "_act")(x)


def build_model(img_size=128, in_ch=3, base_f=32,
                groups=8, dropout=0.25):
    f  = [base_f * (2**i) for i in range(5)]  # [32,64,128,256,512]
    dp = [0.00, 0.05, 0.10, 0.15, 0.20]

    inp = Input(shape=(img_size, img_size, in_ch), name="image")

    e1 = res_block(inp, f[0], groups, dp[0])
    p1 = layers.MaxPooling2D(2)(e1)
    e2 = res_block(p1,  f[1], groups, dp[1])
    p2 = layers.MaxPooling2D(2)(e2)
    e3 = res_block(p2,  f[2], groups, dp[2])
    p3 = layers.MaxPooling2D(2)(e3)
    e4 = res_block(p3,  f[3], groups, dp[3])
    p4 = layers.MaxPooling2D(2)(e4)
    bn = res_block(p4,  f[4], groups, dp[4])
    bn = layers.SpatialDropout2D(dropout)(bn)

    def dec(x, skip, filters, sdp):
        x    = layers.Conv2DTranspose(
            filters, 2, strides=2, padding="same")(x)
        skip = attn_gate(g=x, s=skip, f=filters//2)
        x    = layers.Concatenate()([x, skip])
        return res_block(x, filters, groups, sdp)

    d4 = dec(bn, e4, f[3], dp[3])
    d3 = dec(d4, e3, f[2], dp[2])
    d2 = dec(d3, e2, f[1], dp[1])
    d1 = dec(d2, e1, f[0], dp[0])

    ds2 = ds_head(d3, img_size, "ds_quarter")
    ds1 = ds_head(d2, img_size, "ds_half")

    out = layers.Conv2D(1, 1, padding="same")(d1)
    out = layers.Activation("sigmoid", dtype="float32", name="main")(out)

    return Model(inp, [out, ds1, ds2], name="AttnResUNet_leaky")


# ─────────────────────────────────────────────────────────────────────────────
# 7.  LOSS & METRICS
# ─────────────────────────────────────────────────────────────────────────────
_FT_A = CFG["FT_ALPHA"];   _FT_B = CFG["FT_BETA"]
_FT_G = CFG["FT_GAMMA"];   _BF_G = CFG["BF_GAMMA"]
_FT_W = CFG["LOSS_FT_W"];  _BF_W = CFG["LOSS_BF_W"]

def focal_tversky(y_true, y_pred, smooth=1e-6):
    yt = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    yp = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    TP = tf.reduce_sum(yt * yp)
    FP = tf.reduce_sum((1 - yt) * yp)
    FN = tf.reduce_sum(yt * (1 - yp))
    ti = (TP + smooth) / (TP + _FT_A*FN + _FT_B*FP + smooth)
    return tf.pow(1. - ti, 1. / _FT_G)

def binary_focal(y_true, y_pred, smooth=1e-7):
    yt = tf.cast(y_true, tf.float32)
    yp = tf.clip_by_value(tf.cast(y_pred, tf.float32), smooth, 1.-smooth)
    bce = -(yt*tf.math.log(yp) + (1-yt)*tf.math.log(1-yp))
    pt  = tf.where(tf.equal(yt, 1), yp, 1-yp)
    return tf.reduce_mean(tf.pow(1-pt, _BF_G) * bce)

def compound_loss(y_true, y_pred):
    return _FT_W*focal_tversky(y_true, y_pred) + \
           _BF_W*binary_focal(y_true, y_pred)

def soft_dice(y_true, y_pred, smooth=1e-6):
    if isinstance(y_pred, (list, tuple)): y_pred = y_pred[0]
    yt = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    yp = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    return (2*tf.reduce_sum(yt*yp)+smooth) / \
           (tf.reduce_sum(yt)+tf.reduce_sum(yp)+smooth)

def hard_dice(gt, pred, smooth=1e-6):
    i = np.sum(gt * pred)
    return float((2*i + smooth) / (gt.sum() + pred.sum() + smooth))

def hard_iou(gt, pred, smooth=1e-6):
    i = np.sum(gt * pred)
    u = gt.sum() + pred.sum() - i
    return float((i + smooth) / (u + smooth))


# ─────────────────────────────────────────────────────────────────────────────
# 8.  LR SCHEDULE
# ─────────────────────────────────────────────────────────────────────────────
class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, lr_max, lr_min, warmup_steps, total_steps):
        super().__init__()
        self.lr_max       = float(lr_max)
        self.lr_min       = float(lr_min)
        self.warmup_steps = float(warmup_steps)
        self.total_steps  = float(total_steps)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        wu_lr  = self.lr_max * (step / self.warmup_steps)
        prog   = tf.clip_by_value(
            (step - self.warmup_steps) /
            (self.total_steps - self.warmup_steps), 0., 1.)
        cos_lr = self.lr_min + 0.5*(self.lr_max - self.lr_min) * \
                 (1. + tf.cos(math.pi * prog))
        return tf.where(step < self.warmup_steps, wu_lr, cos_lr)

    def get_config(self):
        return dict(lr_max=self.lr_max, lr_min=self.lr_min,
                    warmup_steps=self.warmup_steps,
                    total_steps=self.total_steps)


# ─────────────────────────────────────────────────────────────────────────────
# 9.  POST-PROCESSING
# ─────────────────────────────────────────────────────────────────────────────
def morph_filter(binary, min_px=20):
    labeled, n = scipy_label(binary)
    out = np.zeros_like(binary)
    for cid in range(1, n+1):
        if (labeled == cid).sum() >= min_px:
            out[labeled == cid] = 1
    return out.astype(np.uint8)


# ─────────────────────────────────────────────────────────────────────────────
# 10.  TRAINING
# ─────────────────────────────────────────────────────────────────────────────
def train(df_tr_raw, df_test):
    print(f"\n{'='*62}")
    print(f"  TRAINING — leaky evaluation")
    print(f"  Train: {len(df_tr_raw)} | Val (leaky test): {len(df_test)}")
    print(f"{'='*62}")

    df_tr = make_balanced_train_df(df_tr_raw)
    ckpt  = os.path.join(CFG["OUTPUT_DIR"], "leaky_best.keras")

    ds_tr = build_dataset(df_tr,   training=True,  batch_size=CFG["BATCH_SIZE"])
    ds_vl = build_dataset(df_test, training=False, batch_size=CFG["BATCH_SIZE"])

    model = build_model(
        CFG["IMG_SIZE"], CFG["INPUT_CHANNELS"],
        CFG["BASE_FILTERS"], CFG["GROUP_NORM_G"], CFG["DROPOUT"])
    model.summary(line_length=85)

    steps_bal    = math.ceil(len(df_tr) / CFG["BATCH_SIZE"])
    total_steps  = CFG["EPOCHS"] * steps_bal
    warmup_steps = CFG["WARMUP_EPOCHS"] * steps_bal

    schedule = WarmupCosineDecay(
        CFG["LR_INITIAL"], CFG["LR_MIN"],
        warmup_steps, total_steps)

    # No LossScaleOptimizer — mixed precision is disabled
    opt = tf.keras.optimizers.AdamW(
        learning_rate=schedule,
        weight_decay=CFG["WEIGHT_DECAY"],
        clipnorm=CFG["GRAD_CLIP"])

    model.compile(
        optimizer=opt,
        loss={
            "main":           compound_loss,
            "ds_half_act":    compound_loss,
            "ds_quarter_act": compound_loss,
        },
        loss_weights={
            "main": 1.0,
            "ds_half_act":    CFG["DS_WEIGHT"],
            "ds_quarter_act": CFG["DS_WEIGHT"],
        },
        metrics={"main": [soft_dice]},
    )

    def wrap(img, msk):
        return img, {"main": msk, "ds_half_act": msk, "ds_quarter_act": msk}

    lr_log = []
    def _log_lr(epoch, logs):
        try:
            inner = (model.optimizer.inner_optimizer
                     if hasattr(model.optimizer, "inner_optimizer")
                     else model.optimizer)
            lr_log.append(float(schedule(float(inner.iterations.numpy()))))
        except:
            lr_log.append(CFG["LR_INITIAL"])

    callbacks = [
        ModelCheckpoint(ckpt, monitor="val_main_soft_dice",
                        mode="max", save_best_only=True, verbose=1),
        EarlyStopping(monitor="val_main_soft_dice", mode="max",
                      patience=CFG["ES_PATIENCE"],
                      restore_best_weights=True, verbose=1),
        CSVLogger(os.path.join(CFG["OUTPUT_DIR"], "training_log.csv")),
        LambdaCallback(on_epoch_end=_log_lr),
    ]

    hist = model.fit(
        ds_tr.map(wrap),
        validation_data=ds_vl.map(wrap),
        epochs=CFG["EPOCHS"],
        callbacks=callbacks,
        verbose=1,
    )

    best_val = max(hist.history.get("val_main_soft_dice", [0.]))
    print(f"\n  Best val Soft Dice (LEAKY): {best_val:.4f}")
    _plot_history(hist, lr_log)
    return model, ckpt


# ─────────────────────────────────────────────────────────────────────────────
# 11.  EVALUATION — slice-by-slice to avoid batch issues
# ─────────────────────────────────────────────────────────────────────────────
def evaluate(model, df_test):
    print(f"\n{'='*62}")
    print(f"  EVALUATION ON LEAKY TEST SET ({len(df_test)} slices)")
    print(f"{'='*62}")

    # Slice-by-slice inference — no batch size issues
    soft_preds, gt_masks = [], []
    for _, row in df_test.iterrows():
        # Load image
        img_raw = cv2.imread(row["image_path"], cv2.IMREAD_COLOR)
        if img_raw is None:
            img_arr = np.zeros((IMG_SIZE, IMG_SIZE, 3), np.float32)
        else:
            img_arr = cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB)
            img_arr = cv2.resize(img_arr, (IMG_SIZE, IMG_SIZE))
            img_arr = img_arr.astype(np.float32) / 255.0
        img_t = tf.constant(img_arr[np.newaxis])  # (1,H,W,3)

        # Load mask
        msk_raw = cv2.imread(row["mask_path"], cv2.IMREAD_GRAYSCALE)
        if msk_raw is None:
            gt = np.zeros((IMG_SIZE, IMG_SIZE), np.float32)
        else:
            gt = cv2.resize(msk_raw, (IMG_SIZE, IMG_SIZE), cv2.INTER_NEAREST)
            gt = (gt > 127).astype(np.float32)

        # Predict
        preds = model(img_t, training=False)
        main  = (preds[0] if isinstance(preds, (list,tuple)) else preds)
        soft  = main.numpy()[0, :, :, 0]  # (H, W)

        soft_preds.append(soft)
        gt_masks.append(gt)

    soft = np.array(soft_preds)   # (N, H, W)
    gt   = np.array(gt_masks)     # (N, H, W)

    # Threshold grid search
    best_t, best_d = 0.50, 0.0
    print("\n  Threshold search:")
    for t in np.arange(0.25, 0.91, 0.025):
        dices = []
        for i in range(len(soft)):
            b = morph_filter((soft[i] >= t).astype(np.uint8),
                             CFG["MIN_COMPONENT_PX"])
            dices.append(hard_dice(gt[i].astype(np.uint8), b))
        md  = float(np.mean(dices))
        tag = " <- best" if md > best_d else ""
        print(f"    t={t:.3f}  Dice={md:.4f}{tag}")
        if md > best_d:
            best_d, best_t = md, t

    print(f"\n  Optimal threshold = {best_t:.3f}  |  Dice = {best_d:.4f}")

    # Final metrics at best threshold
    dices, ious = [], []
    for i in range(len(soft)):
        b = morph_filter((soft[i] >= best_t).astype(np.uint8),
                         CFG["MIN_COMPONENT_PX"])
        dices.append(hard_dice(gt[i].astype(np.uint8), b))
        ious.append(hard_iou(gt[i].astype(np.uint8), b))

    pos_mask = gt.reshape(len(gt), -1).max(1) > 0
    tp_dices = np.array(dices)[pos_mask]

    print(f"\n{'='*62}")
    print(f"  LEAKY EVALUATION RESULTS")
    print(f"  ⚠️  WARNING: Patient-level data leakage present!")
    print(f"{'='*62}")
    print(f"  Threshold              : {best_t:.3f}")
    print(f"  Mean Hard Dice (all)   : {np.mean(dices):.4f}")
    print(f"  Mean Hard IoU  (all)   : {np.mean(ious):.4f}")
    print(f"  Mean Hard Dice (tumor+): {tp_dices.mean():.4f} "
          f"± {tp_dices.std():.4f}")
    print(f"{'='*62}")

    df_out = df_test.copy()
    df_out["hard_dice"] = dices
    df_out["hard_iou"]  = ious
    df_out.to_csv(os.path.join(CFG["OUTPUT_DIR"],
                               "leaky_per_slice.csv"), index=False)

    return dict(
        threshold     = best_t,
        mean_dice_all = float(np.mean(dices)),
        mean_iou_all  = float(np.mean(ious)),
        tp_mean_dice  = float(tp_dices.mean()),
        tp_std_dice   = float(tp_dices.std()),
        dices         = np.array(dices),
        pos_mask      = pos_mask,
    )


# ─────────────────────────────────────────────────────────────────────────────
# 12.  COMPARISON REPORT
# ─────────────────────────────────────────────────────────────────────────────
def print_comparison_report(leaky):
    # Results from v3 (GroupKFold, no leakage) — update after v3 run
    V3 = dict(mean_dice_all=0.8967, mean_iou_all=0.8646, tp_mean_dice=0.7735)

    d_all = leaky["mean_dice_all"] - V3["mean_dice_all"]
    d_iou = leaky["mean_iou_all"]  - V3["mean_iou_all"]
    d_tp  = leaky["tp_mean_dice"]  - V3["tp_mean_dice"]

    report = f"""
{'='*70}
  SCIENTIFIC COMPARISON: DATA LEAKAGE INFLATION ANALYSIS
  LGG Brain MRI (110 patients, ~3929 slices)
  Architecture: Attention Res-UNet 4.8M params — IDENTICAL in both runs
{'='*70}

  METHODOLOGY:
  ┌─────────────────────────┬──────────────────┬─────────────────┐
  │ Criterion               │ Leaky (this run) │ Clean (v3)      │
  ├─────────────────────────┼──────────────────┼─────────────────┤
  │ Split method            │ Random slice     │ GroupKFold      │
  │ Patient grouping        │ ✗ Absent         │ ✓ Strict        │
  │ Train/test patient OL.  │ ✓ ~100%          │ ✗ None (0%)     │
  │ Cross-validation        │ ✗ Single split   │ ✓ 5-fold        │
  │ Comparable to           │ LiteMRINet 2025  │ This paper      │
  └─────────────────────────┴──────────────────┴─────────────────┘

  RESULTS:
  ┌─────────────────────────┬──────────────────┬─────────────────┬──────────┐
  │ Metric                  │ Leaky (inflated) │ Clean (honest)  │ Inflation│
  ├─────────────────────────┼──────────────────┼─────────────────┼──────────┤
  │ Hard Dice (all slices)  │ {leaky["mean_dice_all"]:.4f}           │ {V3["mean_dice_all"]:.4f}          │ +{d_all:.4f} │
  │ Hard IoU  (all slices)  │ {leaky["mean_iou_all"]:.4f}           │ {V3["mean_iou_all"]:.4f}          │ +{d_iou:.4f} │
  │ Hard Dice (tumor+ only) │ {leaky["tp_mean_dice"]:.4f}           │ {V3["tp_mean_dice"]:.4f}          │ +{d_tp:.4f} │
  └─────────────────────────┴──────────────────┴─────────────────┴──────────┘

  vs LiteMRINet (PLOS ONE 2025, random slice split):
  ┌─────────────────────────┬──────────────────┬─────────────────┐
  │ Metric                  │ LiteMRINet       │ Ours (clean)    │
  ├─────────────────────────┼──────────────────┼─────────────────┤
  │ F1 / Dice (all slices)  │ 85.50%           │ {V3["mean_dice_all"]*100:.2f}%         │
  │ mIoU      (all slices)  │ 87.20%           │ {V3["mean_iou_all"]*100:.2f}%         │
  │ Split method            │ Random slice     │ GroupKFold      │
  │ Data leakage            │ Present          │ None            │
  └─────────────────────────┴──────────────────┴─────────────────┘

  KEY FINDING:
  Data leakage inflates Hard Dice by ~{d_all*100:.2f}% (all slices)
  and ~{d_tp*100:.2f}% (tumor-positive slices).

  Our clean-protocol model achieves {V3["mean_dice_all"]*100:.2f}% Dice (all slices),
  EXCEEDING LiteMRINet's 85.50% that used a leaky protocol.
{'='*70}
"""
    print(report)
    p = os.path.join(CFG["OUTPUT_DIR"], "comparison_report.txt")
    with open(p, "w") as f:
        f.write(report)
    print(f"  Report saved: {p}")


# ─────────────────────────────────────────────────────────────────────────────
# 13.  PLOTS
# ─────────────────────────────────────────────────────────────────────────────
def _plot_history(hist, lr_log):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(hist.history["loss"],     label="Train")
    axes[0].plot(hist.history["val_loss"], label="Val (LEAKY)")
    axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)
    k = "main_soft_dice"
    if k in hist.history:
        axes[1].plot(hist.history[k],          label="Train Dice")
        axes[1].plot(hist.history[f"val_{k}"], label="Val Dice (LEAKY)")
    axes[1].set_title("Soft Dice"); axes[1].legend(); axes[1].grid(alpha=0.3)
    axes[2].plot(lr_log, color="darkorange")
    axes[2].set_title("LR (log)"); axes[2].set_yscale("log"); axes[2].grid(alpha=0.3)
    plt.suptitle("⚠️ LEAKY EVALUATION — scientific comparison only",
                 color="red", fontsize=11, fontweight="bold")
    plt.tight_layout()
    p = os.path.join(CFG["OUTPUT_DIR"], "leaky_curves.png")
    plt.savefig(p, dpi=100); plt.close()
    print(f"  Curves: {p}")


def plot_dice_dist(res):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].hist(res["dices"], bins=50, color="steelblue", edgecolor="w")
    ax[0].axvline(res["mean_dice_all"], color="red", lw=2,
                  label=f"Mean={res['mean_dice_all']:.4f}")
    ax[0].set_title("Hard Dice — All Slices (LEAKY)"); ax[0].legend()
    tp_d = res["dices"][res["pos_mask"]]
    ax[1].hist(tp_d, bins=50, color="darkorange", edgecolor="w")
    ax[1].axvline(tp_d.mean(), color="red", lw=2,
                  label=f"Mean={tp_d.mean():.4f}")
    ax[1].set_title("Hard Dice — Tumor+ Only (LEAKY)"); ax[1].legend()
    plt.suptitle("⚠️ LEAKY Evaluation — Data Leakage Present",
                 color="red", fontsize=12, fontweight="bold")
    plt.tight_layout()
    p = os.path.join(CFG["OUTPUT_DIR"], "leaky_dice_dist.png")
    plt.savefig(p, dpi=110); plt.close()
    print(f"  Histogram: {p}")


# ─────────────────────────────────────────────────────────────────────────────
# 14.  MAIN
# ─────────────────────────────────────────────────────────────────────────────
def main():
    print("\n" + "="*62)
    print("  LGG Segmentation — LEAKY EVALUATION")
    print("  Purpose: Quantify data leakage metric inflation")
    print("  ⚠️  Controlled experiment — NOT the primary result")
    print("="*62 + "\n")

    df = build_dataframe(CFG["DATA_ROOT"], CFG["ZIP_PATH"])
    df_train, df_test = make_leaky_split(df)
    model, ckpt = train(df_train, df_test)
    results = evaluate(model, df_test)
    plot_dice_dist(results)
    print_comparison_report(results)
    print("\n  Done. See outputs_leaky/ for all results.\n")


if __name__ == "__main__":
    main()

2026-05-12 06:49:56.981994: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778568597.001376    5157 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778568597.006611    5157 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-12 06:49:57.028531: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Mixed precision: DISABLED (float32 throughout)

  LGG Segmentation — LEAKY EVALUATION
  Purpose: Quantify data leakage metric inflation
  ⚠️  Controlled experiment — NOT the primary result

Dataset: 3929 slices | 1373 tumor+ | 110 patients

  LEAKY SPLIT STATISTICS (replicating published papers)
  Train slices : 3001
  Test slices  : 928
  Total patients        : 110
  Patients train only   : 0
  Patients test only    : 0
  *** LEAKED patients   : 110 ***
  *** Leakage rate      : 100.0% of all patients ***

  Leakage report: ./outputs_leaky/leakage_report.txt

  TRAINING — leaky evaluation
  Train: 3001 | Val (leaky test): 928
  Balanced train: 2875 slices (tumor 72.8%)


I0000 00:00:1778568600.170850    5157 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22161 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:00:06.0, compute capability: 8.9


Model: "AttnResUNet_leaky"

┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)           ┃ Output Shape        ┃     Param # ┃ Connected to         ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)     │ (None, 128, 128, 3) │           0 │ -                    │
├────────────────────────┼─────────────────────┼─────────────┼──────────────────────┤
│ conv2d_1 (Conv2D)      │ (None, 128, 128,    │         864 │ image[0][0]          │
│                        │ 32)                 │             │                      │
├────────────────────────┼─────────────────────┼─────────────┼──────────────────────┤
│ group_norm_1           │ (None, 128, 128,    │          64 │ conv2d_1[0][0]       │
│ (GroupNorm)            │ 32)                 │             │                      │
├────────────────────────┼─────────────────────┼─────────────┼──────────────────────┤
│ activation             │ (None, 128, 128,    │           0 │ group_norm_1[0][0]   │
│ (Activation)           │ 32)                 │             │                      │
├────────────────────────┼─────────────────────┼─────────────┼──────────────────────┤
│ conv2d_2 (Conv2D)      │ (None, 128, 128,    │       9,216 │ activation[0][0]     │
│                        │ 32)                 │             │                      │
├────────────────────────┼─────────────────────┼─────────────┼──────────────────────┤
│ conv2d (Conv2D)        │ (None, 128, 128,    │          96 │ image[0][0]          │
│                        │ 32)                 │             │                      │
├────────────────────────┼─────────────────────┼─────────────┼──────────────────────┤
│ group_norm_2           │ (None, 128, 128,    │          64 │ conv2d_2[0][0]       │
│ (GroupNorm)            │ 32)                 │             │                      │
├────────────────────────┼─────────────────────┼─────────────┼──────────────────────┤
│ group_norm (GroupNorm) │ (None, 128, 128,    │          64 │ conv2d[0][0]         │
│                        │ 32)                 │             │                      │
├────────────────────────┼─────────────────────┼─────────────┼──────────────────────┤
│ add (Add)              │ (None, 128, 128,    │           0 │ group_norm_2[0][0],  │
│                        │ 32)                 │             │ group_norm[0][0]     │
├────────────────────────┼─────────────────────┼─────────────┼──────────────────────┤
│ activation_1           │ (None, 128, 128,    │           0 │ add[0][0]            │
│ (Activation)           │ 32)                 │             │                      │
├────────────────────────┼─────────────────────┼─────────────┼──────────────────────┤
│ max_pooling2d          │ (None, 64, 64, 32)  │           0 │ activation_1[0][0]   │
│ (MaxPooling2D)         │                     │             │                      │
├────────────────────────┼─────────────────────┼─────────────┼──────────────────────┤
│ conv2d_4 (Conv2D)      │ (None, 64, 64, 64)  │      18,432 │ max_pooling2d[0][0]  │
├────────────────────────┼─────────────────────┼─────────────┼──────────────────────┤
│ group_norm_4           │ (None, 64, 64, 64)  │         128 │ conv2d_4[0][0]       │
│ (GroupNorm)            │                     │             │                      │
├────────────────────────┼─────────────────────┼─────────────┼──────────────────────┤
│ activation_2           │ (None, 64, 64, 64)  │           0 │ group_norm_4[0][0]   │
│ (Activation)           │                     │             │                      │
├────────────────────────┼─────────────────────┼─────────────┼──────────────────────┤
│ conv2d_5 (Conv2D)      │ (None, 64, 64, 64)  │      36,864 │ activation_2[0][0]   │
├────────────────────────┼─────────────────────┼─────────────┼──────────────────────┤
│ group_norm_5           │ (None, 64, 64, 64)  │         128 │ conv2d_5[0][0]       │
│ (GroupNorm)            │                  

...

Epoch 1/120


I0000 00:00:1778568625.190143    5289 service.cc:148] XLA service 0x7ba8c40023d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778568625.190191    5289 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2026-05-12 06:50:26.230954: W tensorflow/compiler/tf2xla/kernels/random_ops.cc:61] Warning: Using tf.random.uniform with XLA compilation will ignore seeds; consider using tf.random.stateless_uniform instead if reproducible behavior is desired. AttnResUNet_leaky_1/stochastic_depth_1/random_uniform/RandomUniform
2026-05-12 06:50:26.237593: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778568630.293474    5289 cuda_dnn.cc:529] Loaded cuDNN version 90501


  6/179 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - ds_half_act_loss: 0.6008 - ds_quarter_act_loss: 0.6224 - loss: 1.0404 - main_loss: 0.6734 - main_soft_dice: 0.0755    

I0000 00:00:1778568656.923785    5289 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


179/179 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - ds_half_act_loss: 0.5579 - ds_quarter_act_loss: 0.5459 - loss: 0.9367 - main_loss: 0.6055 - main_soft_dice: 0.0907
Epoch 1: val_main_soft_dice improved from -inf to 0.20263, saving model to ./outputs_leaky/leaky_best.keras
179/179 ━━━━━━━━━━━━━━━━━━━━ 79s 142ms/step - ds_half_act_loss: 0.5575 - ds_quarter_act_loss: 0.5454 - loss: 0.9359 - main_loss: 0.6050 - main_soft_dice: 0.0910 - val_ds_half_act_loss: 0.3255 - val_ds_quarter_act_loss: 0.3242 - val_loss: 0.5512 - val_main_loss: 0.3563 - val_main_soft_dice: 0.2026
Epoch 2/120
179/179 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - ds_half_act_loss: 0.3227 - ds_quarter_act_loss: 0.3228 - loss: 0.5170 - main_loss: 0.3233 - main_soft_dice: 0.3560 
Epoch 2: val_main_soft_dice improved from 0.20263 to 0.51872, saving model to ./outputs_leaky/leaky_best.keras
179/179 ━━━━━━━━━━━━━━━━━━━━ 21s 117ms/step - ds_half_act_loss: 0.3226 - ds_quarter_act_loss: 0.3227 - loss: 0.5168 - main_loss: 0.3232 - main_soft_dice